# OVERVIEW

## Structura dataset

setul de date este in `/kaggle/input/competitions/rsna-intracranial-aneurysm-detection/`, care contine 5 elemente:

| Element | Continut |
|---|---|
| `segmentations/` | 356 de fisiere NIfTI (numarul asta e dat de prezenta ambelor masti: anevrisme si vase de sange) |
| `series/` | 4348 de foldere, cate unul pentru fiecare serie din `train.csv`. Numele folderului este `SeriesInstanceUID`, iar in interior sunt fisierele DICOM (`.dcm`) ale seriei. |
| `train.csv` | O linie pentru fiecare serie |
| `train_localizers.csv` | Pozitia+info fiecarui anevrism din seriile pozitive |
| `kaggle_evaluation/` cred ca este irelevant pentru mine

## Subsetul cu segmentari

Folderul `segmentations/` contine 356 de fisiere NIfTI, grupate in perechi dupa `SeriesInstanceUID`:
- `<SeriesInstanceUID>.nii`: volumul 3D al seriei ;
- `<SeriesInstanceUID>_cowseg.nii`: masca vaselor, cu segmentarea a cercului lui willis (paper, pag. 10).


### Verificari

| Verificare | Rezultat |
|---|---|
| Perechi imagine + masca | 178 din 178 complete |
| Serii care nu apar in `train.csv` | 1 din 178 (['1.2.826.0.1.3680043.8.498.43502795339700498960289295234851562632']) nu are eticheta |
| Serii cu segmentare si eticheta | 177,  143 cu anevrism (81%) |
| Anevrisme marcate in `train_localizers.csv` | 188 de centroizi marcati; 34 de serii au mai mult de un anevrism |

### Pe modalitate

| Modalitate | Fara anevrism | Cu anevrism | Total |
|---|---|---|---|
| CTA | 7 | 67 | 74 |
| MRA | 4 | 37 | 41 |
| MRI T1post | 5 | 15 | 20 |
| MRI T2 | 18 | 24 | 42 |
| **Total** | **34** | **143** | **177** |

### Pe locatie 

| Locatie | Serii |
|---|---|
| Anterior Communicating Artery | 36 |
| Right Supraclinoid ICA | 22 |
| Left Supraclinoid ICA | 22 |
| Right Middle Cerebral Artery | 20 |
| Other Posterior Circulation | 19 |
| Basilar Tip | 13 |
| Right Anterior Cerebral Artery | 9 |
| Left Infraclinoid ICA | 7 |
| Right Posterior Communicating Artery | 7 |
| Left Posterior Communicating Artery | 7 |
| Left Middle Cerebral Artery | 6 |
| Right Infraclinoid ICA | 4 |
| Left Anterior Cerebral Artery | 1 |

Suma pe locatii (173) e mai mica decat numarul de puncte (188). (probabil ca daca exista 2> anevrisme in acelasi loc in train.cvs se considera doar una???)

### Observatii
- Subsetul are o rata de pozitive mai mare decat setul total (train.csv): 81%, fata de 43%

- Pe CTA si MRA, exista doar 115 .nii (74 + 41) si alea puternic imbalansate.

- Tot subsetul de segmentare se afla in setul de antrenare al challenge-ului (paper, pag. 21).

## Etichetele din mastile de vase

Celula citeste fiecare masca `_cowseg.nii` si afiseaza valorile distincte din ea (etichetele).

### Rezultate
- Toate cele 178 de masti sunt `int16`, cu valori de la 0 (fundal) la 13.
- Fiecare eticheta 1–13 corespunde uneia dintre cele 13 locatii arteriale ale challenge-ului (p11).
- Nu toate mastile contin toate etichetele: doar 50 din 178 au toate cele 13.

| Eticheta | Locatie (ipoteza, vezi mai jos) | Masti in care apare (din 178) |
|---|---|---|
| 1 | Other Posterior Circulation | 178 |
| 2 | Basilar Tip | 173 |
| 3 | Right Posterior Communicating Artery | 107 |
| 4 | Left Posterior Communicating Artery | 76 |
| 5 | Right Infraclinoid ICA | 177 |
| 6 | Left Infraclinoid ICA | 173 |
| 7 | Right Supraclinoid ICA | 178 |
| 8 | Left Supraclinoid ICA | 178 |
| 9 | Right Middle Cerebral Artery | 178 |
| 10 | Left Middle Cerebral Artery | 178 |
| 11 | Right Anterior Cerebral Artery | 178 |
| 12 | Left Anterior Cerebral Artery | 178 |
| 13 | Anterior Communicating Artery | 147 |

## Exemplu de header DICOM

Celula citeste headerul unui singur fisier DICOM, fara pixeli. Fisierele sunt denumite dupa `SOPInstanceUID`, asa ca ordinea lor alfabetica nu e ordinea feliilor in serie. Ordinea reala se stabileste dupa pozitia feliei (pag. 10).

| Grup | Tag | Folosit pentru |
|---|---|---|
| Centru | `(0013,1001)` (bloc privat `RSNA`) | ID-ul centrului, coloana `center` |
| | `FrameOfReferenceUID` | coloana `FoR`; contine ID-ul centrului|
| Achizitie | `Manufacturer` | producatorul, grupat in coloana `vendor` |
| Geometrie | `SliceThickness` | grosimea nominala a feliei (filtrul MRA ≤ 2 mm, analiza de shortcut-uri) |
| | `ImagePositionPatient`, `ImageOrientationPatient`, `InstanceNumber` | pasul dintre felii, `z_step`: pozitia fiecarei felii e proiectata pe normala la felie (produsul vectorial al celor doi vectori de orientare), iar diferenta de pozitie se imparte la diferenta de `InstanceNumber` |
| | `PixelSpacing` | rezolutia in plan|
| | `SharedFunctionalGroupsSequence`, `PerFrameFunctionalGroupsSequence` | aceleasi campuri geometrice la seriile Philips multi-frame |
| Pacient | `SmokingStatus` | statutul de fumator (lipseste la aproape toate seriile) |

### Observatii
- Grosimea nominala `SliceThickness` e de doua ori mai mare decat distanta dintre felii `SpacingBetweenSlices`, deci feliile se suprapun.

## Modalitati

Numarul de serii pe modalitate, din `train.csv`:

| Modalitate | Serii | Cu anevrism | Fara anevrism |
|---|---|---|---|
| CTA | 1808 | 973 | 835 |
| MRA | 1252 | 555 | 697 |
| MRI T2 | 983 | 258 | 725 |
| MRI T1post | 305 | 77 | 228 |

## Tabelul de protocol


| Coloana | Continut |
|---|---|
| `center` | ID-ul centrului din tagul `(0013,1001)`, sau `unknown` |
| `FoR` | `FrameOfReferenceUID` |
| `n_files` | numarul de fisiere DICOM din serie |
| `SliceThickness` | grosimea nominala a feliei (mm) |
| `z_step` | pasul real dintre felii (mm), masurat din pozitia feliilor |
| `PixelSpacing` | rezolutia in plan (mm) |
| `Manufacturer`, `vendor` | producatorul, grupat (Siemens, Philips, GE, Canon/Toshiba, Hitachi) |
| `Model`, `FieldStrength`, `SeriesDescription` | modelul aparatului, campul magnetic (MR) si descrierea seriei; |

## Z-spacing

Grosimea nominala (`SliceThickness`) si spacing-ul pe z (`z_step`), valori in mm.

| Modalitate | Eticheta | Grosime mediana | Grosime medie ± std | Grosime min–max | Spacing mediana | Spacing medie ± std | Spacing min–max |
|---|---|---|---|---|---|---|---|
| CTA | cu anevrism | 0.80 | 1.10 ± 1.02 | 0.47–10.20 | 0.62 | 0.95 ± 1.01 | 0.25–7.00 |
| | fara anevrism | 0.80 | 1.33 ± 1.41 | 0.50–10.20 | 0.62 | 1.16 ± 1.43 | 0.25–10.00 |
| MRA | cu anevrism | 0.60 | 0.82 ± 0.39 | 0.30–2.00 | 0.55 | 0.61 ± 0.14 | 0.30–1.10 |
| | fara anevrism | 0.60 | 0.83 ± 0.43 | 0.30–5.00 | 0.60 | 0.61 ± 0.21 | 0.30–5.00 |
| MRI T1post | cu anevrism | 1.30 | 1.75 ± 1.28 | 0.80–5.00 | 1.00 | 1.62 ± 1.62 | 0.50–7.00 |
| | fara anevrism | 1.20 | 2.21 ± 1.60 | 0.80–5.00 | 1.00 | 2.34 ± 2.06 | 0.50–7.00 |
| MRI T2 | cu anevrism | 5.00 | 4.28 ± 1.17 | 0.50–6.00 | 5.00 | 4.76 ± 1.47 | 0.50–7.00 |
| | fara anevrism | 5.00 | 4.29 ± 1.02 | 0.50–6.00 | 5.00 | 4.82 ± 1.39 | 0.40–7.00 |

## Producatori

Numarul de serii pe producator (`Manufacturer`, exact cum apare in header), pentru CTA si MRA:

| Modalitate | Manufacturer | Fara anevrism | Cu anevrism |
|---|---|---|---|
| CTA | Canon Medical Systems | 7 | 10 |
| | GE MEDICAL SYSTEMS | 344 | 268 |
| | Philips | 122 | 154 |
| | SIEMENS | 202 | 284 |
| | Siemens Healthineers | 24 | 24 |
| | TOSHIBA | 136 | 233 |
| MRA | CANON_MEC | 15 | 2 |
| | GE MEDICAL SYSTEMS | 158 | 140 |
| | Hitachi, Ltd. | 1 | 0 |
| | Philips | 23 | 7 |
| | Philips Healthcare | 10 | 5 |
| | Philips Medical Systems | 85 | 94 |
| | SIEMENS | 206 | 175 |
| | Siemens | 31 | 14 |
| | Siemens HealthCare GmbH | 0 | 2 |
| | Siemens Healthineers | 136 | 95 |
| | TOSHIBA | 29 | 19 |
| | TOSHIBA_MEC | 3 | 2 |



## Producator si rata de pozitive

Seriile grupate pe producator (`vendor`) si proportia celor cu anevrism (`pos_rate`), pentru CTA si MRA:

| Modalitate | Vendor | Fara anevrism | Cu anevrism | pos_rate |
|---|---|---|---|---|
| CTA | Canon/Toshiba | 143 | 243 | 0.63 |
| | GE | 344 | 268 | 0.44 |
| | Philips | 122 | 154 | 0.56 |
| | Siemens | 226 | 308 | 0.58 |
| MRA | Canon/Toshiba | 47 | 23 | 0.33 |
| | GE | 158 | 140 | 0.47 |
| | Hitachi | 1 | 0 | 0.00 |
| | Philips | 118 | 106 | 0.47 |
| | Siemens | 373 | 286 | 0.43 |

- La CTA, rata de pozitive variaza intre 0.44 (GE) si 0.63 (Canon/Toshiba)
- La MRA, ratele sunt apropiate (0.43–0.47), cu exceptia Canon/Toshiba (0.33, doar 70 de serii). 

## Centre

Centrul fiecarei serii e ID-ul din tagul privat `(0013,1001)`. Seriile fara acest tag sunt marcate `unknown`.

| Centru | Serii (toate modalitatile) | CTA fara | CTA cu | CTA pos_rate | MRA fara | MRA cu | MRA pos_rate |
|---|---|---|---|---|---|---|---|
| unknown | 1992 | 412 | 544 | 0.57 | 287 | 213 | 0.43 |
| 966396 | 391 | 74 | 82 | 0.53 | 70 | 35 | 0.33 |
| 968599 | 323 | – | – | – | 77 | 80 | 0.51 |
| 963218 | 242 | 47 | 82 | 0.64 | 33 | 23 | 0.41 |
| 963661 | 241 | 70 | 54 | 0.44 | 46 | 66 | 0.59 |
| 961671 | 236 | 51 | 40 | 0.44 | 17 | 15 | 0.47 |
| 962005 | 229 | 7 | 11 | 0.61 | 75 | 51 | 0.40 |
| 961598 | 198 | 43 | 14 | 0.25 | 37 | 37 | 0.50 |
| 966514 | 193 | 65 | 86 | 0.57 | 17 | 9 | 0.35 |
| 961932 | 92 | 32 | 60 | 0.65 | – | – | – |
| 961980 | 49 | – | – | – | 8 | 9 | 0.53 |
| 961601 | 41 | 34 | 0 | 0.00 | 4 | 1 | 0.20 |
| 961976 | 37 | – | – | – | 6 | 6 | 0.50 |
| 962507 | 27 | – | – | – | 5 | 5 | 0.50 |
| 962183 | 24 | – | – | – | 7 | 1 | 0.12 |
| 962274 | 24 | – | – | – | 7 | 2 | 0.22 |
| 962508 | 9 | – | – | – | 1 | 2 | 0.67 |

## Protocol pe centru

Pe fiecare centru, modalitate si eticheta: numarul de serii, producatorii, spacing-ul in plan (mediana, mm) si grosimea nominala (mediana si min–max, mm). Apar doar CTA si MRA si doar grupurile cu cel putin 10 serii.

| Centru | Modalitate | Eticheta | Serii | Vendor | Spacing in plan | Grosime mediana (min–max) |
|---|---|---|---|---|---|---|
| 961598 | CTA | cu anevrism | 14 | GE | 0.48 | 5.00 (5.00–5.00) |
| | | fara anevrism | 43 | GE | 0.49 | 5.00 (5.00–5.00) |
| | MRA | cu anevrism | 37 | GE | 0.43 | 1.40 (1.20–1.60) |
| | | fara anevrism | 37 | GE | 0.43 | 1.60 (1.20–1.60) |
| 961601 | CTA | fara anevrism | 34 | GE | 0.52 | 5.00 (5.00–5.00) |
| 961671 | CTA | cu anevrism | 40 | GE | 0.49 | 0.62 (0.62–1.25) |
| | | fara anevrism | 51 | GE | 0.49 | 0.62 (0.62–1.25) |
| | MRA | cu anevrism | 15 | GE, Philips | 0.43 | 1.40 (0.60–2.00) |
| | | fara anevrism | 17 | GE, Philips | 0.43 | 1.40 (0.60–1.40) |
| 961932 | CTA | cu anevrism | 60 | Canon/Toshiba, GE, Philips, Siemens | 0.44 | 0.67 (0.50–0.80) |
| | | fara anevrism | 32 | Philips, Siemens | 0.45 | 0.75 (0.60–3.00) |
| 962005 | CTA | cu anevrism | 11 | GE, Siemens | 0.46 | 0.75 (0.60–1.00) |
| | MRA | cu anevrism | 51 | GE, Siemens | 0.35 | 0.50 (0.30–1.60) |
| | | fara anevrism | 75 | GE, Siemens | 0.35 | 0.50 (0.30–1.60) |
| 963218 | CTA | cu anevrism | 82 | Canon/Toshiba, GE, Philips, Siemens | 0.43 | 0.62 (0.50–5.00) |
| | | fara anevrism | 47 | Canon/Toshiba, GE, Siemens | 0.42 | 0.62 (0.50–3.00) |
| | MRA | cu anevrism | 23 | GE, Siemens | 0.37 | 0.52 (0.50–2.00) |
| | | fara anevrism | 33 | GE, Siemens | 0.43 | 0.52 (0.40–2.00) |
| 963661 | CTA | cu anevrism | 54 | Canon/Toshiba, Philips, Siemens | 0.46 | 1.00 (0.50–3.00) |
| | | fara anevrism | 70 | Canon/Toshiba, Philips, Siemens | 0.54 | 1.00 (0.50–1.00) |
| | MRA | cu anevrism | 66 | Philips, Siemens | 0.35 | 0.50 (0.40–1.80) |
| | | fara anevrism | 46 | Siemens | 0.35 | 0.50 (0.40–0.65) |
| 966396 | CTA | cu anevrism | 82 | Canon/Toshiba, GE, Philips, Siemens | 0.45 | 1.00 (0.50–2.00) |
| | | fara anevrism | 74 | Canon/Toshiba, Philips, Siemens | 0.46 | 1.00 (0.60–1.00) |
| | MRA | cu anevrism | 35 | GE, Siemens | 0.26 | 0.50 (0.50–1.00) |
| | | fara anevrism | 70 | Siemens | 0.35 | 0.50 (0.50–0.80) |
| 966514 | CTA | cu anevrism | 86 | Canon/Toshiba, GE | 0.43 | 3.00 (0.50–10.20) |
| | | fara anevrism | 65 | Canon/Toshiba, GE | 0.45 | 3.00 (0.50–10.20) |
| | MRA | fara anevrism | 17 | Canon/Toshiba, GE | 0.41 | 1.20 (0.90–1.20) |
| 968599 | MRA | cu anevrism | 80 | Philips | 0.50 | 0.55 (0.55–0.55) |
| | | fara anevrism | 77 | Philips | 0.50 | 0.55 (0.55–5.00) |
| unknown | CTA | cu anevrism | 544 | Canon/Toshiba, GE, Philips, Siemens | 0.48 | 0.80 (0.47–4.00) |
| | | fara anevrism | 412 | Canon/Toshiba, GE, Philips, Siemens | 0.47 | 0.62 (0.50–4.00) |
| | MRA | cu anevrism | 213 | Canon/Toshiba, GE, Philips, Siemens | 0.43 | 1.00 (0.50–2.00) |
| | | fara anevrism | 287 | Canon/Toshiba, GE, Hitachi, Philips, Siemens | 0.43 | 0.85 (0.40–3.00) |

## Shortcut



### Pe tot setul

| Variabila | CTA AUC | CTA putere | MRA AUC | MRA putere |
|---|---|---|---|---|
| Grosime nominala | 0.492 | 0.508 | 0.501 | 0.501 |
| Spacing z (`z_step`) | 0.521 | 0.521 | 0.495 | 0.505 |
| Spacing in plan | 0.425 | 0.575 | 0.456 | 0.544 |
| Varsta | 0.514 | 0.514 | 0.588 | 0.588 |
| Sex (femeie) | 0.573 | 0.573 | 0.567 | 0.567 |
| Centru | 0.560 | 0.560 | 0.535 | 0.535 |
| Producator | 0.564 | 0.564 | 0.497 | 0.503 |

### In interiorul unui centru

doar centrele cu cel putin 10 serii in fiecare clasa si doar variabilele cu putere ≥ 0.65.

| Centru | Modalitate | Variabila | Serii | AUC | Putere |
|---|---|---|---|---|---|
| 961671 | MRA | Varsta | 32 | 0.884 | 0.884 |
| 961671 | CTA | Varsta | 91 | 0.824 | 0.824 |
| 966514 | CTA | Spacing in plan | 151 | 0.216 | 0.784 |
| 963661 | CTA | Spacing in plan | 124 | 0.269 | 0.731 |
| 963661 | MRA | Grosime nominala | 112 | 0.684 | 0.684 |
| 963661 | MRA | Spacing z | 112 | 0.666 | 0.666 |
| 963218 | CTA | Varsta | 129 | 0.659 | 0.659 |

## Varsta, sex, fumat

### Varsta (ani)

| Modalitate | Eticheta | Serii | Medie ± std | Mediana | Min–max |
|---|---|---|---|---|---|
| CTA | fara anevrism | 835 | 58.4 ± 17.1 | 60 | 18–89 |
| | cu anevrism | 973 | 59.8 ± 14.4 | 60 | 21–89 |
| MRA | fara anevrism | 697 | 55.5 ± 16.3 | 57 | 18–89 |
| | cu anevrism | 555 | 60.6 ± 14.0 | 62 | 20–89 |

### Sex

| Modalitate | Eticheta | Femei | Barbati | % femei |
|---|---|---|---|---|
| CTA | fara anevrism | 479 | 356 | 57.4 |
| | cu anevrism | 701 | 272 | 72.0 |
| MRA | fara anevrism | 457 | 240 | 65.6 |
| | cu anevrism | 438 | 117 | 78.9 |

### Fumat
Tagul `SmokingStatus` lipseste la 4034 din 4348 de serii; la 289 are valoarea `UNKNOWN`, iar la 25 e gol. 

## AUC pentru varsta si sex



| Modalitate | Varsta AUC | Sex (femeie) AUC |
|---|---|---|
| CTA | 0.514 | 0.573 |
| MRA | 0.588 | 0.567 |
| MRI T1post | 0.596 | 0.504 |
| MRI T2 | 0.558 | 0.565 |
| Toate | 0.549 | 0.556 |


## Comparatie pozitive / negative in interiorul centrelor

Pentru fiecare centru si modalitate: valoarea la seriile cu anevrism / fara anevrism. Apar doar CTA si MRA si doar centrele cu cel putin 10 serii in fiecare clasa. Grosimea si spacing-ul sunt mediane, in mm.

| Centru | Modalitate | Varsta medie | % femei | Grosime | Spacing z | Spacing in plan |
|---|---|---|---|---|---|---|
| 961598 | CTA | 66.0 / 62.2 | 42.9 / 48.8 | 5.00 / 5.00 | 5.00 / 5.00 | 0.48 / 0.49 |
| 961671 | CTA | **61.2 / 40.0** | 82.5 / 68.6 | 0.62 / 0.62 | 0.62 / 0.62 | 0.49 / 0.49 |
| 961932 | CTA | 58.9 / 61.5 | 68.3 / 46.9 | 0.67 / 0.75 | 0.40 / 0.50 | 0.44 / 0.45 |
| 963218 | CTA | 58.7 / 51.2 | 78.0 / 76.6 | 0.62 / 0.62 | 0.62 / 0.62 | 0.43 / 0.42 |
| 963661 | CTA | 69.2 / 67.5 | 72.2 / 52.9 | 1.00 / 1.00 | 0.50 / 0.50 | 0.46 / 0.54 |
| 966396 | CTA | 58.8 / 60.1 | 75.6 / 56.8 | 1.00 / 1.00 | 0.70 / 0.70 | 0.45 / 0.46 |
| 966514 | CTA | 49.0 / 50.3 | 58.1 / 40.0 | 3.00 / 3.00 | 3.00 / 3.00 | 0.43 / 0.45 |
| unknown | CTA | 60.6 / 60.0 | 73.2 / 60.4 | 0.80 / 0.62 | 0.62 / 0.50 | 0.48 / 0.47 |
| 961598 | MRA | 60.6 / 54.6 | **81.1 / 43.2** | 1.40 / 1.60 | 0.70 / 0.80 | 0.43 / 0.43 |
| 961671 | MRA | **64.3 / 41.4** | 73.3 / 52.9 | 1.40 / 1.40 | 0.70 / 0.70 | 0.43 / 0.43 |
| 962005 | MRA | 62.9 / 56.0 | 76.5 / 69.3 | 0.50 / 0.50 | 0.50 / 0.50 | 0.35 / 0.35 |
| 963218 | MRA | 61.9 / 54.3 | 69.6 / 69.7 | 0.52 / 0.52 | 0.52 / 0.52 | 0.37 / 0.43 |
| 963661 | MRA | 64.3 / 59.5 | 80.3 / 52.2 | 0.50 / 0.50 | 0.50 / 0.50 | 0.35 / 0.35 |
| 966396 | MRA | 57.7 / 55.9 | 74.3 / 68.6 | 0.50 / 0.50 | 0.50 / 0.50 | 0.26 / 0.35 |
| 968599 | MRA | 57.0 / 55.1 | 83.8 / 70.1 | 0.55 / 0.55 | 0.55 / 0.55 | 0.50 / 0.50 |
| unknown | MRA | 60.7 / 55.7 | 79.8 / 69.0 | 1.00 / 0.85 | 0.65 / 0.70 | 0.43 / 0.43 |

## Etichete si localizari (CTA + MRA)

Celula verifica daca cele trei surse de etichete spun acelasi lucru si descrie anevrismele din seriile pozitive:
- `Aneurysm Present` din `train.csv` (eticheta seriei);
- cele 13 coloane de locatie din `train.csv` (0/1 pe locatie);
- punctele din `train_localizers.csv` (cate un punct pentru fiecare anevrism, cu felia, coordonatele si locatia).

### Consistenta etichetelor

| Verificare | Serii |
|---|---|
| Pozitive fara nicio locatie in `train.csv` | 0 |
| Negative cu locatie in `train.csv` | 0 |
| Pozitive fara punct in `train_localizers.csv` | 0 |
| Negative cu punct in `train_localizers.csv` | 0 |
| Puncte cu o locatie nemarcata in `train.csv` | 1 |

### Anevrisme pe serie (doar pozitive)

| Modalitate | 1 | 2 | 3 | 4+ | Total |
|---|---|---|---|---|---|
| CTA | 783 | 145 | 36 | 9 | 973 |
| MRA | 479 | 55 | 15 | 6 | 555 |
| Total | 1262 | 200 | 51 | 15 | 1528 |

### Serii pozitive pe locatie

O serie poate avea anevrisme la mai multe locatii, deci suma pe locatii (1816) depaseste numarul de serii pozitive (1528).

| Locatie | CTA | MRA | Total | % |
|---|---|---|---|---|
| Anterior Communicating Artery | 232 | 81 | 313 | 17.2 |
| Left Supraclinoid ICA | 145 | 123 | 268 | 14.8 |
| Right Middle Cerebral Artery | 161 | 80 | 241 | 13.3 |
| Right Supraclinoid ICA | 104 | 121 | 225 | 12.4 |
| Left Middle Cerebral Artery | 130 | 56 | 186 | 10.2 |
| Right Posterior Communicating Artery | 73 | 20 | 93 | 5.1 |
| Basilar Tip | 65 | 25 | 90 | 5.0 |
| Other Posterior Circulation | 65 | 25 | 90 | 5.0 |
| Right Infraclinoid ICA | 46 | 36 | 82 | 4.5 |
| Left Posterior Communicating Artery | 62 | 13 | 75 | 4.1 |
| Left Infraclinoid ICA | 33 | 32 | 65 | 3.6 |
| Right Anterior Cerebral Artery | 35 | 12 | 47 | 2.6 |
| Left Anterior Cerebral Artery | 32 | 9 | 41 | 2.3 |

In [ ]:
import os

COMP = "/kaggle/input/competitions/rsna-intracranial-aneurysm-detection"

def show(path):
    print(f"\n=== {path}")
    if not os.path.isdir(path):
        print("not found")
        return
    for name in sorted(os.listdir(path)):
        p = os.path.join(path, name)
        if os.path.isdir(p):
            inside = sorted(os.listdir(p))
            
            print(f"[folder] {name}/  {len(inside)} elem. \n ex: {inside}" if name not in ("segmentations","series") else f"[folder] {name}/  {len(inside)} elem.\n ex: {inside[:5]}" )
        else:
            print(f"\n[file] {name}  {os.path.getsize(p) / 1e6:.1f} MB")

for path in ("/kaggle/input", "/kaggle/input/competitions", COMP, "/kaggle/working"):
    show(path)

## Structura dataset

setul de date este in `/kaggle/input/competitions/rsna-intracranial-aneurysm-detection/`, care contine 5 elemente:

| Element | Continut |
|---|---|
| `segmentations/` | 356 de fisiere NIfTI (numarul asta e dat de prezenta ambelor masti: anevrisme si vase de sange) |
| `series/` | 4348 de foldere, cate unul pentru fiecare serie din `train.csv`. Numele folderului este `SeriesInstanceUID`, iar in interior sunt fisierele DICOM (`.dcm`) ale seriei. |
| `train.csv` | O linie pentru fiecare serie |
| `train_localizers.csv` | Pozitia+info fiecarui anevrism din seriile pozitive |
| `kaggle_evaluation/` cred ca este irelevant pentru mine

In [ ]:
import pandas as pd
import os
SEG = "/kaggle/input/competitions/rsna-intracranial-aneurysm-detection/segmentations"
COMP = "/kaggle/input/competitions/rsna-intracranial-aneurysm-detection"

rows = []
def aggregate():
    rows = []
    for f in os.listdir(SEG):
        is_willis_mask = f.endswith("_cowseg.nii")
        rows.append(
            {
                "SeriesInstanceUID": f[:-len("_cowseg.nii")] if is_willis_mask else f[:-len(".nii")],
                "kind": "mask" if is_willis_mask else "image",
                "path": os.path.join(SEG, f)
            }
        )
    print(pd.DataFrame(rows)["kind"].value_counts())
    pairs = pd.DataFrame(rows).pivot(index="SeriesInstanceUID", columns="kind", values="path").reset_index()
    print(f"found: {len(pairs)} complete: {pairs[['image', 'mask']].notna().all(axis=1).sum()}")
    print(f"missing Willis: {pairs[['mask']].isna().all(axis=1).sum()}")
    print(f"missing segm_mask: {pairs[['mask']].isna().all(axis=1).sum()}")
    train_file = pd.read_csv('/kaggle/input/competitions/rsna-intracranial-aneurysm-detection/train.csv')
    # for row in rows:
    #     res = train_file.isin([row['SeriesInstanceUID']]).any().any()
    #     if res == False:
    #         print("something is missing")
    missing = pairs[~pairs["SeriesInstanceUID"].isin(train_file["SeriesInstanceUID"])]
    print("\n\n")
    print(f"not in train.csv: {len(missing)} din {len(pairs)}")
    print(missing["SeriesInstanceUID"].tolist())
    print("\n\n")

    train = pd.read_csv(os.path.join(COMP, "train.csv"))
    loc = pd.read_csv(os.path.join(COMP, "train_localizers.csv"))
    LOC_COLS = [c for c in train.columns if c not in ("SeriesInstanceUID", "PatientAge", "PatientSex", "Modality", "Aneurysm Present")]
    
    seg = train[train["SeriesInstanceUID"].isin(pairs["SeriesInstanceUID"])]
    print(f"total: {len(seg)} | positive: {int(seg['Aneurysm Present'].sum())}")
    print(pd.crosstab(seg["Modality"], seg["Aneurysm Present"], margins=True))
    print("\n\n")

    n = loc[loc["SeriesInstanceUID"].isin(seg["SeriesInstanceUID"])].groupby("SeriesInstanceUID").size()
    print(f"\naneurysm in train_localizers.csv: {int(n.sum())} | more than one aneurysm: {int((n > 1).sum())}")
    print("\ndistrib:\n")
    print(f"{seg[LOC_COLS].sum().sort_values(ascending=False).to_string()}")

    

    

    
        
        
        

#seg()
aggregate()

## Subsetul cu segmentari

Folderul `segmentations/` contine 356 de fisiere NIfTI, grupate in perechi dupa `SeriesInstanceUID`:
- `<SeriesInstanceUID>.nii`: volumul 3D al seriei ;
- `<SeriesInstanceUID>_cowseg.nii`: masca vaselor, cu segmentarea a cercului lui willis (paper, pag. 10).


### Verificari

| Verificare | Rezultat |
|---|---|
| Perechi imagine + masca | 178 din 178 complete |
| Serii care nu apar in `train.csv` | 1 din 178 (['1.2.826.0.1.3680043.8.498.43502795339700498960289295234851562632']) nu are eticheta |
| Serii cu segmentare si eticheta | 177,  143 cu anevrism (81%) |
| Anevrisme marcate in `train_localizers.csv` | 188 de centroizi marcati; 34 de serii au mai mult de un anevrism |

### Pe modalitate

| Modalitate | Fara anevrism | Cu anevrism | Total |
|---|---|---|---|
| CTA | 7 | 67 | 74 |
| MRA | 4 | 37 | 41 |
| MRI T1post | 5 | 15 | 20 |
| MRI T2 | 18 | 24 | 42 |
| **Total** | **34** | **143** | **177** |

### Pe locatie 

| Locatie | Serii |
|---|---|
| Anterior Communicating Artery | 36 |
| Right Supraclinoid ICA | 22 |
| Left Supraclinoid ICA | 22 |
| Right Middle Cerebral Artery | 20 |
| Other Posterior Circulation | 19 |
| Basilar Tip | 13 |
| Right Anterior Cerebral Artery | 9 |
| Left Infraclinoid ICA | 7 |
| Right Posterior Communicating Artery | 7 |
| Left Posterior Communicating Artery | 7 |
| Left Middle Cerebral Artery | 6 |
| Right Infraclinoid ICA | 4 |
| Left Anterior Cerebral Artery | 1 |

Suma pe locatii (173) e mai mica decat numarul de puncte (188). (probabil ca daca exista 2> anevrisme in acelasi loc in train.cvs se considera doar una???)

### Observatii
- Subsetul are o rata de pozitive mai mare decat setul total (train.csv): 81%, fata de 43%

- Pe CTA si MRA, exista doar 115 .nii (74 + 41) si alea puternic imbalansate.

- Tot subsetul de segmentare se afla in setul de antrenare al challenge-ului (paper, pag. 21).

In [ ]:
SEG = "/kaggle/input/competitions/rsna-intracranial-aneurysm-detection/segmentations"
import pandas as pd
import nibabel as nib
import numpy as np
import os
def define_type_seg():
    counts = {}
    for sample in os.listdir(SEG):
        if not sample.endswith("_cowseg.nii"):
            continue
        data = np.asarray(nib.load(os.path.join(SEG, sample)).dataobj)
        labels = np.unique(data)
        for label in labels.tolist():
            counts[label] = counts.get(label, 0) + 1
        print(sample[0:30], data.dtype, labels.tolist())
    print("\nditrib labels in masti:", dict(sorted(counts.items())))
        

define_type_seg()


## Etichetele din mastile de vase

Celula citeste fiecare masca `_cowseg.nii` si afiseaza valorile distincte din ea (etichetele).

### Rezultate
- Toate cele 178 de masti sunt `int16`, cu valori de la 0 (fundal) la 13.
- Fiecare eticheta 1–13 corespunde uneia dintre cele 13 locatii arteriale ale challenge-ului (p11).
- Nu toate mastile contin toate etichetele: doar 50 din 178 au toate cele 13.

| Eticheta | Locatie (ipoteza, vezi mai jos) | Masti in care apare (din 178) |
|---|---|---|
| 1 | Other Posterior Circulation | 178 |
| 2 | Basilar Tip | 173 |
| 3 | Right Posterior Communicating Artery | 107 |
| 4 | Left Posterior Communicating Artery | 76 |
| 5 | Right Infraclinoid ICA | 177 |
| 6 | Left Infraclinoid ICA | 173 |
| 7 | Right Supraclinoid ICA | 178 |
| 8 | Left Supraclinoid ICA | 178 |
| 9 | Right Middle Cerebral Artery | 178 |
| 10 | Left Middle Cerebral Artery | 178 |
| 11 | Right Anterior Cerebral Artery | 178 |
| 12 | Left Anterior Cerebral Artery | 178 |
| 13 | Anterior Communicating Artery | 147 |




In [ ]:
import os, pydicom

SERIES = "/kaggle/input/competitions/rsna-intracranial-aneurysm-detection/series"
uid = sorted(os.listdir(SERIES))[1]
folder = os.path.join(SERIES, uid)
first = sorted(os.listdir(folder))[1]

ds = pydicom.dcmread(os.path.join(folder, first), stop_before_pixels=True)
print(uid, "|", len(os.listdir(folder)), "series")
print(ds)

## Exemplu de header DICOM

Celula citeste headerul unui singur fisier DICOM, fara pixeli. Fisierele sunt denumite dupa `SOPInstanceUID`, asa ca ordinea lor alfabetica nu e ordinea feliilor in serie. Ordinea reala se stabileste dupa pozitia feliei (pag. 10).

| Grup | Tag | Folosit pentru |
|---|---|---|
| Centru | `(0013,1001)` (bloc privat `RSNA`) | ID-ul centrului, coloana `center` |
| | `FrameOfReferenceUID` | coloana `FoR`; contine ID-ul centrului|
| Achizitie | `Manufacturer` | producatorul, grupat in coloana `vendor` |
| Geometrie | `SliceThickness` | grosimea nominala a feliei (filtrul MRA ≤ 2 mm, analiza de shortcut-uri) |
| | `ImagePositionPatient`, `ImageOrientationPatient`, `InstanceNumber` | pasul dintre felii, `z_step`: pozitia fiecarei felii e proiectata pe normala la felie (produsul vectorial al celor doi vectori de orientare), iar diferenta de pozitie se imparte la diferenta de `InstanceNumber` |
| | `PixelSpacing` | rezolutia in plan|
| | `SharedFunctionalGroupsSequence`, `PerFrameFunctionalGroupsSequence` | aceleasi campuri geometrice la seriile Philips multi-frame |
| Pacient | `SmokingStatus` | statutul de fumator (lipseste la aproape toate seriile) |

### Observatii
- Grosimea nominala `SliceThickness` e de doua ori mai mare decat distanta dintre felii `SpacingBetweenSlices`, deci feliile se suprapun.


In [ ]:

import os 
import pydicom 
import pandas as pd
TRAIN_CSV = "/kaggle/input/competitions/rsna-intracranial-aneurysm-detection/train.csv"
train = pd.read_csv(TRAIN_CSV)

df = train["Modality"].value_counts().rename_axis("Modality").reset_index(name="n")
df["n_pos"] = df["Modality"].map(train.groupby("Modality")["Aneurysm Present"].sum())
df["n_neg"] = df["n"] - df["n_pos"]
print(df)


## Modalitati

Numarul de serii pe modalitate, din `train.csv`:

| Modalitate | Serii | Cu anevrism | Fara anevrism |
|---|---|---|---|
| CTA | 1808 | 973 | 835 |
| MRA | 1252 | 555 | 697 |
| MRI T2 | 983 | 258 | 725 |
| MRI T1post | 305 | 77 | 228 |



In [ ]:
import os, pydicom
import pandas as pd

COMP = "/kaggle/input/competitions/rsna-intracranial-aneurysm-detection"
SERIES = os.path.join(COMP, "series")
OUT = "/kaggle/working"
train = pd.read_csv(os.path.join(COMP, "train.csv"))

In [ ]:
import numpy as np

VENDORS = {"siemens": "Siemens", "philips": "Philips", "ge ": "GE", "toshiba": "Canon/Toshiba", "canon": "Canon/Toshiba", "hitachi": "Hitachi"}


def get_attr(ds, name):
    if name in ds:
        return ds.get(name)
    for seq in ("SharedFunctionalGroupsSequence", "PerFrameFunctionalGroupsSequence"):
        if seq in ds and "PixelMeasuresSequence" in getattr(ds, seq)[0]:
            pm = getattr(ds, seq)[0].PixelMeasuresSequence[0]
            if name in pm:
                return pm.get(name)
    return None


def slice_step(folder, files, ds):
    iop = ds.get("ImageOrientationPatient")
    if iop is None and ds.get("SharedFunctionalGroupsSequence") and "PlaneOrientationSequence" in ds.SharedFunctionalGroupsSequence[0]:
        iop = ds.SharedFunctionalGroupsSequence[0].PlaneOrientationSequence[0].ImageOrientationPatient
    if iop is None:
        return None
    normal = np.cross(np.array(iop[:3], float), np.array(iop[3:], float))
    frames = ds.get("PerFrameFunctionalGroupsSequence")
    if frames and len(frames) > 1:
        z = np.sort([np.dot(np.array(f.PlanePositionSequence[0].ImagePositionPatient, float), normal) for f in frames if "PlanePositionSequence" in f])
        steps = np.diff(z)[np.diff(z) > 1e-3]
        return round(float(np.median(steps)), 3) if len(steps) else None
    picks = [ds] + [pydicom.dcmread(os.path.join(folder, f), stop_before_pixels=True) for f in files[1:3]]
    pts = [(np.dot(np.array(p.ImagePositionPatient, float), normal), int(p.InstanceNumber)) for p in picks
           if p.get("ImagePositionPatient") is not None and p.get("InstanceNumber") is not None]
    est = [abs(a[0] - b[0]) / abs(a[1] - b[1]) for i, a in enumerate(pts) for b in pts[i + 1:] if a[1] != b[1]]
    return round(float(np.median(est)), 3) if est else None


def series_header(uid):
    folder = os.path.join(SERIES, uid)
    files = sorted(os.listdir(folder))
    ds = pydicom.dcmread(os.path.join(folder, files[0]), stop_before_pixels=True)
    st, ps = get_attr(ds, "SliceThickness"), get_attr(ds, "PixelSpacing")
    try:
        site = ds.private_block(0x0013, "RSNA")[0x01].value
        if isinstance(site, bytes):
            site = site.decode().strip("\x00 ")
    except KeyError:
        site = "unknown"
    return {"SeriesInstanceUID": uid,
            "center": site,
            "FoR": str(ds.get("FrameOfReferenceUID", "")),
            "n_files": len(files),
            "SliceThickness": float(st) if st is not None else None,
            "z_step": slice_step(folder, files, ds),
            "PixelSpacing": float(ps[0]) if ps is not None else None,
            "Manufacturer": ds.get("Manufacturer"),
            "Model": ds.get("ManufacturerModelName"),
            "FieldStrength": ds.get("MagneticFieldStrength"),
            "SeriesDescription": ds.get("SeriesDescription")
           }


def vendor_of(manufacturer):
    m = str(manufacturer or "").lower()
    return next((v for k, v in VENDORS.items() if k in m), "unknown")


def pos_rate_table(df, col):
    t = pd.crosstab([df["Modality"], df[col]], df["Aneurysm Present"])
    t["pos_rate"] = (t[1] / t.sum(axis=1)).round(2)
    return t

In [ ]:
proto = train[["SeriesInstanceUID", "Modality", "Aneurysm Present"]].merge(pd.DataFrame([series_header(uid) for uid in train["SeriesInstanceUID"]]), on="SeriesInstanceUID")
proto["vendor"] = proto["Manufacturer"].map(vendor_of)
proto.to_csv(os.path.join(OUT, "rsna_protocol.csv"), index=False)
print(len(proto), "series")

## Tabelul de protocol


| Coloana | Continut |
|---|---|
| `center` | ID-ul centrului din tagul `(0013,1001)`, sau `unknown` |
| `FoR` | `FrameOfReferenceUID` |
| `n_files` | numarul de fisiere DICOM din serie |
| `SliceThickness` | grosimea nominala a feliei (mm) |
| `z_step` | pasul real dintre felii (mm), masurat din pozitia feliilor |
| `PixelSpacing` | rezolutia in plan (mm) |
| `Manufacturer`, `vendor` | producatorul, asa cum apare in header si grupat (Siemens, Philips, GE, Canon/Toshiba, Hitachi) |
| `Model`, `FieldStrength`, `SeriesDescription` | modelul aparatului, campul magnetic (MR) si descrierea seriei; salvate pentru referinta, neanalizate |



In [ ]:
def summary(x):
    return f"{x.median():.2f} | {x.mean():.2f} ± {x.std():.2f} | {x.min():.2f}-{x.max():.2f}"

s = proto.assign(label=proto["Aneurysm Present"].map({0: "fara anevrism", 1: "cu anevrism"}))
tab = s.groupby(["Modality", "label"]).agg(grosime_nominala=("SliceThickness", summary), pas_real=("z_step", summary))
print("valori in mm: mediana | medie ± std | min-max\n")
print(tab.to_string())
print("\nz_step lipsa:", proto["z_step"].isna().sum())

## Z-spacing

Grosimea nominala (`SliceThickness`) si spacing-ul pe z (`z_step`), valori in mm.

| Modalitate | Eticheta | Grosime mediana | Grosime medie ± std | Grosime min–max | Spacing mediana | Spacing medie ± std | Spacing min–max |
|---|---|---|---|---|---|---|---|
| CTA | cu anevrism | 0.80 | 1.10 ± 1.02 | 0.47–10.20 | 0.62 | 0.95 ± 1.01 | 0.25–7.00 |
| | fara anevrism | 0.80 | 1.33 ± 1.41 | 0.50–10.20 | 0.62 | 1.16 ± 1.43 | 0.25–10.00 |
| MRA | cu anevrism | 0.60 | 0.82 ± 0.39 | 0.30–2.00 | 0.55 | 0.61 ± 0.14 | 0.30–1.10 |
| | fara anevrism | 0.60 | 0.83 ± 0.43 | 0.30–5.00 | 0.60 | 0.61 ± 0.21 | 0.30–5.00 |
| MRI T1post | cu anevrism | 1.30 | 1.75 ± 1.28 | 0.80–5.00 | 1.00 | 1.62 ± 1.62 | 0.50–7.00 |
| | fara anevrism | 1.20 | 2.21 ± 1.60 | 0.80–5.00 | 1.00 | 2.34 ± 2.06 | 0.50–7.00 |
| MRI T2 | cu anevrism | 5.00 | 4.28 ± 1.17 | 0.50–6.00 | 5.00 | 4.76 ± 1.47 | 0.50–7.00 |
| | fara anevrism | 5.00 | 4.29 ± 1.02 | 0.50–6.00 | 5.00 | 4.82 ± 1.39 | 0.40–7.00 |



In [ ]:
print(pd.crosstab([proto["Modality"], proto["Manufacturer"]], proto["Aneurysm Present"]))

## Producatori

Numarul de serii pe producator (`Manufacturer`, exact cum apare in header), pentru CTA si MRA:

| Modalitate | Manufacturer | Fara anevrism | Cu anevrism |
|---|---|---|---|
| CTA | Canon Medical Systems | 7 | 10 |
| | GE MEDICAL SYSTEMS | 344 | 268 |
| | Philips | 122 | 154 |
| | SIEMENS | 202 | 284 |
| | Siemens Healthineers | 24 | 24 |
| | TOSHIBA | 136 | 233 |
| MRA | CANON_MEC | 15 | 2 |
| | GE MEDICAL SYSTEMS | 158 | 140 |
| | Hitachi, Ltd. | 1 | 0 |
| | Philips | 23 | 7 |
| | Philips Healthcare | 10 | 5 |
| | Philips Medical Systems | 85 | 94 |
| | SIEMENS | 206 | 175 |
| | Siemens | 31 | 14 |
| | Siemens HealthCare GmbH | 0 | 2 |
| | Siemens Healthineers | 136 | 95 |
| | TOSHIBA | 29 | 19 |
| | TOSHIBA_MEC | 3 | 2 |

- Acelasi producator apare sub mai multe denumiri (de exemplu `SIEMENS`, `Siemens`, `Siemens Healthineers`, `Siemens HealthCare GmbH`). De aceea denumirile sunt grupate in coloana `vendor`: Siemens, Philips, GE, Canon/Toshiba, Hitachi.


In [ ]:
print(pos_rate_table(proto, "vendor"))

## Producator si rata de pozitive

Seriile grupate pe producator (`vendor`) si proportia celor cu anevrism (`pos_rate`), pentru CTA si MRA:

| Modalitate | Vendor | Fara anevrism | Cu anevrism | pos_rate |
|---|---|---|---|---|
| CTA | Canon/Toshiba | 143 | 243 | 0.63 |
| | GE | 344 | 268 | 0.44 |
| | Philips | 122 | 154 | 0.56 |
| | Siemens | 226 | 308 | 0.58 |
| MRA | Canon/Toshiba | 47 | 23 | 0.33 |
| | GE | 158 | 140 | 0.47 |
| | Hitachi | 1 | 0 | 0.00 |
| | Philips | 118 | 106 | 0.47 |
| | Siemens | 373 | 286 | 0.43 |

- La CTA, rata de pozitive variaza intre 0.44 (GE) si 0.63 (Canon/Toshiba)
- La MRA, ratele sunt apropiate (0.43–0.47), cu exceptia Canon/Toshiba (0.33, doar 70 de serii). 


In [ ]:
df = proto["center"].value_counts().rename_axis("Centre").reset_index(name="nr")
print(df)
print(f"total:  {df['nr'].sum()}")
print(pos_rate_table(proto, "center"))

## Centre

Centrul fiecarei serii e ID-ul din tagul privat `(0013,1001)`. Seriile fara acest tag sunt marcate `unknown`.

| Centru | Serii (toate modalitatile) | CTA fara | CTA cu | CTA pos_rate | MRA fara | MRA cu | MRA pos_rate |
|---|---|---|---|---|---|---|---|
| unknown | 1992 | 412 | 544 | 0.57 | 287 | 213 | 0.43 |
| 966396 | 391 | 74 | 82 | 0.53 | 70 | 35 | 0.33 |
| 968599 | 323 | – | – | – | 77 | 80 | 0.51 |
| 963218 | 242 | 47 | 82 | 0.64 | 33 | 23 | 0.41 |
| 963661 | 241 | 70 | 54 | 0.44 | 46 | 66 | 0.59 |
| 961671 | 236 | 51 | 40 | 0.44 | 17 | 15 | 0.47 |
| 962005 | 229 | 7 | 11 | 0.61 | 75 | 51 | 0.40 |
| 961598 | 198 | 43 | 14 | 0.25 | 37 | 37 | 0.50 |
| 966514 | 193 | 65 | 86 | 0.57 | 17 | 9 | 0.35 |
| 961932 | 92 | 32 | 60 | 0.65 | – | – | – |
| 961980 | 49 | – | – | – | 8 | 9 | 0.53 |
| 961601 | 41 | 34 | 0 | 0.00 | 4 | 1 | 0.20 |
| 961976 | 37 | – | – | – | 6 | 6 | 0.50 |
| 962507 | 27 | – | – | – | 5 | 5 | 0.50 |
| 962183 | 24 | – | – | – | 7 | 1 | 0.12 |
| 962274 | 24 | – | – | – | 7 | 2 | 0.22 |
| 962508 | 9 | – | – | – | 1 | 2 | 0.67 |




In [ ]:
def protocol_table(df, col):
    d = df.assign(label=df["Aneurysm Present"].map({0: "healthy", 1: "aneurysm"}))
    return d.groupby([col, "Modality", "label"]).agg(series=("SeriesInstanceUID", "size"), vendor=("vendor", lambda v: ", ".join(sorted(v.unique()))),
                                                     spacing_x_median=("PixelSpacing", "median"), slice_median=("SliceThickness", "median"),
                                                     slice_min=("SliceThickness", "min"), slice_max=("SliceThickness", "max")).round(2).reset_index()

tab = protocol_table(proto, "center")
print(tab.to_string(index=False))

print("\n\n\n")
print("doar mra,cta series>10")
print("\n\n\n")

print(tab[tab["Modality"].isin(["CTA", "MRA"]) & (tab["series"] >= 10)].to_string(index=False))

## Protocol pe centru

Pe fiecare centru, modalitate si eticheta: numarul de serii, producatorii, spacing-ul in plan (mediana, mm) si grosimea nominala (mediana si min–max, mm). Apar doar CTA si MRA si doar grupurile cu cel putin 10 serii.

| Centru | Modalitate | Eticheta | Serii | Vendor | Spacing in plan | Grosime mediana (min–max) |
|---|---|---|---|---|---|---|
| 961598 | CTA | cu anevrism | 14 | GE | 0.48 | 5.00 (5.00–5.00) |
| | | fara anevrism | 43 | GE | 0.49 | 5.00 (5.00–5.00) |
| | MRA | cu anevrism | 37 | GE | 0.43 | 1.40 (1.20–1.60) |
| | | fara anevrism | 37 | GE | 0.43 | 1.60 (1.20–1.60) |
| 961601 | CTA | fara anevrism | 34 | GE | 0.52 | 5.00 (5.00–5.00) |
| 961671 | CTA | cu anevrism | 40 | GE | 0.49 | 0.62 (0.62–1.25) |
| | | fara anevrism | 51 | GE | 0.49 | 0.62 (0.62–1.25) |
| | MRA | cu anevrism | 15 | GE, Philips | 0.43 | 1.40 (0.60–2.00) |
| | | fara anevrism | 17 | GE, Philips | 0.43 | 1.40 (0.60–1.40) |
| 961932 | CTA | cu anevrism | 60 | Canon/Toshiba, GE, Philips, Siemens | 0.44 | 0.67 (0.50–0.80) |
| | | fara anevrism | 32 | Philips, Siemens | 0.45 | 0.75 (0.60–3.00) |
| 962005 | CTA | cu anevrism | 11 | GE, Siemens | 0.46 | 0.75 (0.60–1.00) |
| | MRA | cu anevrism | 51 | GE, Siemens | 0.35 | 0.50 (0.30–1.60) |
| | | fara anevrism | 75 | GE, Siemens | 0.35 | 0.50 (0.30–1.60) |
| 963218 | CTA | cu anevrism | 82 | Canon/Toshiba, GE, Philips, Siemens | 0.43 | 0.62 (0.50–5.00) |
| | | fara anevrism | 47 | Canon/Toshiba, GE, Siemens | 0.42 | 0.62 (0.50–3.00) |
| | MRA | cu anevrism | 23 | GE, Siemens | 0.37 | 0.52 (0.50–2.00) |
| | | fara anevrism | 33 | GE, Siemens | 0.43 | 0.52 (0.40–2.00) |
| 963661 | CTA | cu anevrism | 54 | Canon/Toshiba, Philips, Siemens | 0.46 | 1.00 (0.50–3.00) |
| | | fara anevrism | 70 | Canon/Toshiba, Philips, Siemens | 0.54 | 1.00 (0.50–1.00) |
| | MRA | cu anevrism | 66 | Philips, Siemens | 0.35 | 0.50 (0.40–1.80) |
| | | fara anevrism | 46 | Siemens | 0.35 | 0.50 (0.40–0.65) |
| 966396 | CTA | cu anevrism | 82 | Canon/Toshiba, GE, Philips, Siemens | 0.45 | 1.00 (0.50–2.00) |
| | | fara anevrism | 74 | Canon/Toshiba, Philips, Siemens | 0.46 | 1.00 (0.60–1.00) |
| | MRA | cu anevrism | 35 | GE, Siemens | 0.26 | 0.50 (0.50–1.00) |
| | | fara anevrism | 70 | Siemens | 0.35 | 0.50 (0.50–0.80) |
| 966514 | CTA | cu anevrism | 86 | Canon/Toshiba, GE | 0.43 | 3.00 (0.50–10.20) |
| | | fara anevrism | 65 | Canon/Toshiba, GE | 0.45 | 3.00 (0.50–10.20) |
| | MRA | fara anevrism | 17 | Canon/Toshiba, GE | 0.41 | 1.20 (0.90–1.20) |
| 968599 | MRA | cu anevrism | 80 | Philips | 0.50 | 0.55 (0.55–0.55) |
| | | fara anevrism | 77 | Philips | 0.50 | 0.55 (0.55–5.00) |
| unknown | CTA | cu anevrism | 544 | Canon/Toshiba, GE, Philips, Siemens | 0.48 | 0.80 (0.47–4.00) |
| | | fara anevrism | 412 | Canon/Toshiba, GE, Philips, Siemens | 0.47 | 0.62 (0.50–4.00) |
| | MRA | cu anevrism | 213 | Canon/Toshiba, GE, Philips, Siemens | 0.43 | 1.00 (0.50–2.00) |
| | | fara anevrism | 287 | Canon/Toshiba, GE, Hitachi, Philips, Siemens | 0.43 | 0.85 (0.40–3.00) |



In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold


def auc_power(y, x):
    ok = pd.notna(x)
    if ok.sum() < 10 or pd.Series(y[ok]).nunique() < 2:
        return None, None
    auc = roc_auc_score(y[ok], x[ok])
    return round(auc, 3), round(max(auc, 1 - auc), 3)


def oof_rate(df, col, k=5, seed=42):
    rate = pd.Series(np.nan, index=df.index)
    for tr, te in KFold(k, shuffle=True, random_state=seed).split(df):
        rate.iloc[te] = df.iloc[te][col].map(df.iloc[tr].groupby(col)["Aneurysm Present"].mean()).values
    return rate

d = proto.merge(train[["SeriesInstanceUID", "PatientAge", "PatientSex"]], on="SeriesInstanceUID")
d["female"] = (d["PatientSex"] == "Female").astype(int)

rows = []
for mod in ("CTA", "MRA"):
    m = d[d["Modality"] == mod].reset_index(drop=True)
    y = m["Aneurysm Present"].values
    for name, x in [("slice thickness", m["SliceThickness"]), ("slice step (z)", m["z_step"]), ("in-plane spacing", m["PixelSpacing"]), ("age", m["PatientAge"]),
                    ("sex (female)", m["female"]), ("center alone", oof_rate(m, "center")), ("vendor alone", oof_rate(m, "vendor"))]:
        auc, power = auc_power(y, x.values)
        rows.append({"modality": mod, "shortcut": name, "auc": auc, "power": power})
print(pd.DataFrame(rows).to_string(index=False))

rows = []
for (c, mod), m in d[d["Modality"].isin(["CTA", "MRA"])].groupby(["center", "Modality"]):
    if m["Aneurysm Present"].value_counts().reindex([0, 1], fill_value=0).min() < 10:
        continue
    for name, col in [("slice thickness", "SliceThickness"), ("slice step (z)", "z_step"), ("in-plane spacing", "PixelSpacing"), ("age", "PatientAge")]:
        auc, power = auc_power(m["Aneurysm Present"].values, m[col].values)
        rows.append({"center": c, "modality": mod, "shortcut": name, "n": len(m), "auc": auc, "power": power})
w = pd.DataFrame(rows)
print("\n=== in interiorul fiecarui centru (putere >= 0.65)\n", w[w["power"] >= 0.65].sort_values("power", ascending=False).to_string(index=False))

## Shortcut



### Pe tot setul

| Variabila | CTA AUC | CTA putere | MRA AUC | MRA putere |
|---|---|---|---|---|
| Grosime nominala | 0.492 | 0.508 | 0.501 | 0.501 |
| Spacing z (`z_step`) | 0.521 | 0.521 | 0.495 | 0.505 |
| Spacing in plan | 0.425 | 0.575 | 0.456 | 0.544 |
| Varsta | 0.514 | 0.514 | 0.588 | 0.588 |
| Sex (femeie) | 0.573 | 0.573 | 0.567 | 0.567 |
| Centru | 0.560 | 0.560 | 0.535 | 0.535 |
| Producator | 0.564 | 0.564 | 0.497 | 0.503 |

### In interiorul unui centru

Aici apar doar centrele cu cel putin 10 serii in fiecare clasa si doar variabilele cu putere ≥ 0.65.

| Centru | Modalitate | Variabila | Serii | AUC | Putere |
|---|---|---|---|---|---|
| 961671 | MRA | Varsta | 32 | 0.884 | 0.884 |
| 961671 | CTA | Varsta | 91 | 0.824 | 0.824 |
| 966514 | CTA | Spacing in plan | 151 | 0.216 | 0.784 |
| 963661 | CTA | Spacing in plan | 124 | 0.269 | 0.731 |
| 963661 | MRA | Grosime nominala | 112 | 0.684 | 0.684 |
| 963661 | MRA | Spacing z | 112 | 0.666 | 0.666 |
| 963218 | CTA | Varsta | 129 | 0.659 | 0.659 |



In [ ]:
import os, pydicom
import pandas as pd

COMP = "/kaggle/input/competitions/rsna-intracranial-aneurysm-detection"
SERIES = os.path.join(COMP, "series")
d = pd.read_csv(os.path.join(COMP, "train.csv"))[["SeriesInstanceUID", "Modality", "Aneurysm Present", "PatientAge", "PatientSex"]]

smoking = []
for uid in d["SeriesInstanceUID"]:
    folder = os.path.join(SERIES, uid)
    first = sorted(os.listdir(folder))[0]
    ds = pydicom.dcmread(os.path.join(folder, first), stop_before_pixels=True)
    smoking.append(ds.get("SmokingStatus"))
d["Smoking"] = smoking

print("=== varsta")
print(d.groupby(["Modality", "Aneurysm Present"])["PatientAge"].agg(["count", "min", "max", "mean", "median", "std"]).round(1))

print("\n=== sex")
sex = pd.crosstab([d["Modality"], d["Aneurysm Present"]], d["PatientSex"])
sex["female_pct"] = (sex.get("Female", 0) / sex.sum(axis=1) * 100).round(1)
print(sex)

print("\n=== fumator")
print(pd.crosstab([d["Modality"], d["Aneurysm Present"]], d["Smoking"].fillna("lipsa"), margins=True))

## Varsta, sex, fumat

### Varsta (ani)

| Modalitate | Eticheta | Serii | Medie ± std | Mediana | Min–max |
|---|---|---|---|---|---|
| CTA | fara anevrism | 835 | 58.4 ± 17.1 | 60 | 18–89 |
| | cu anevrism | 973 | 59.8 ± 14.4 | 60 | 21–89 |
| MRA | fara anevrism | 697 | 55.5 ± 16.3 | 57 | 18–89 |
| | cu anevrism | 555 | 60.6 ± 14.0 | 62 | 20–89 |

### Sex

| Modalitate | Eticheta | Femei | Barbati | % femei |
|---|---|---|---|---|
| CTA | fara anevrism | 479 | 356 | 57.4 |
| | cu anevrism | 701 | 272 | 72.0 |
| MRA | fara anevrism | 457 | 240 | 65.6 |
| | cu anevrism | 438 | 117 | 78.9 |

### Fumat
Tagul `SmokingStatus` lipseste la 4034 din 4348 de serii; la 289 are valoarea `UNKNOWN`, iar la 25 e gol. 

In [ ]:
from sklearn.metrics import roc_auc_score

print("\nAUC ")
rows = []
for mod, m in list(d.groupby("Modality")) + [("toate", d)]:
    y = m["Aneurysm Present"]
    for name, x in [("varsta", m["PatientAge"]), ("sex (femeie)", (m["PatientSex"] == "Female").astype(int))]:
        auc = roc_auc_score(y, x)
        rows.append({"modality": mod, "variabila": name, "auc": round(auc, 3), "putere": round(max(auc, 1 - auc), 3)})
print(pd.DataFrame(rows).to_string(index=False))

## AUC pentru varsta si sex

AUC-ul obtinut daca varsta sau sexul sunt folosite singure ca predictor al etichetei (0.5 = nicio informatie, putere = max(AUC, 1 − AUC)).

| Modalitate | Varsta AUC | Sex (femeie) AUC |
|---|---|---|
| CTA | 0.514 | 0.573 |
| MRA | 0.588 | 0.567 |
| MRI T1post | 0.596 | 0.504 |
| MRI T2 | 0.558 | 0.565 |
| Toate | 0.549 | 0.556 |

- Toate valorile sunt intre 0.50 si 0.60, deci varsta si sexul prezic slab eticheta la nivel global.
- Toate AUC-urile sunt peste 0.5, asa ca puterea e egala cu AUC-ul.

In [ ]:
def protocol_tab(df, col):
    d = df.assign(label=df["Aneurysm Present"].map({0: "healthy", 1: "aneurysm"}))
    return d.groupby([col, "Modality", "label"]).agg(series=("SeriesInstanceUID", "size"), vendor=("vendor", lambda v: ", ".join(sorted(v.unique()))),
                                                     age_mean=("PatientAge", "mean"), female_pct=("PatientSex", lambda s: (s == "Female").mean() * 100),
                                                     spacing_x_median=("PixelSpacing", "median"), slice_median=("SliceThickness", "median"),
                                                     slice_min=("SliceThickness", "min"), slice_max=("SliceThickness", "max"),
                                                     step_median=("z_step", "median")).round(2).reset_index()

pd_demo = proto.merge(train[["SeriesInstanceUID", "PatientAge", "PatientSex"]], on="SeriesInstanceUID")
tab = protocol_tab(pd_demo, "center")
tab.to_csv(os.path.join(OUT, "rsna_centers_protocol.csv"), index=False)
print(tab.to_string(index=False))

cmp = tab.pivot_table(index=["center", "Modality"], columns="label", values=["age_mean", "female_pct", "slice_median", "step_median", "spacing_x_median"])
print(cmp.round(1).to_string())

## Comparatie pozitive / negative in interiorul centrelor

Pentru fiecare centru si modalitate: valoarea la seriile cu anevrism / fara anevrism. Apar doar CTA si MRA si doar centrele cu cel putin 10 serii in fiecare clasa. Grosimea si spacing-ul sunt mediane, in mm.

| Centru | Modalitate | Varsta medie | % femei | Grosime | Spacing z | Spacing in plan |
|---|---|---|---|---|---|---|
| 961598 | CTA | 66.0 / 62.2 | 42.9 / 48.8 | 5.00 / 5.00 | 5.00 / 5.00 | 0.48 / 0.49 |
| 961671 | CTA | **61.2 / 40.0** | 82.5 / 68.6 | 0.62 / 0.62 | 0.62 / 0.62 | 0.49 / 0.49 |
| 961932 | CTA | 58.9 / 61.5 | 68.3 / 46.9 | 0.67 / 0.75 | 0.40 / 0.50 | 0.44 / 0.45 |
| 963218 | CTA | 58.7 / 51.2 | 78.0 / 76.6 | 0.62 / 0.62 | 0.62 / 0.62 | 0.43 / 0.42 |
| 963661 | CTA | 69.2 / 67.5 | 72.2 / 52.9 | 1.00 / 1.00 | 0.50 / 0.50 | 0.46 / 0.54 |
| 966396 | CTA | 58.8 / 60.1 | 75.6 / 56.8 | 1.00 / 1.00 | 0.70 / 0.70 | 0.45 / 0.46 |
| 966514 | CTA | 49.0 / 50.3 | 58.1 / 40.0 | 3.00 / 3.00 | 3.00 / 3.00 | 0.43 / 0.45 |
| unknown | CTA | 60.6 / 60.0 | 73.2 / 60.4 | 0.80 / 0.62 | 0.62 / 0.50 | 0.48 / 0.47 |
| 961598 | MRA | 60.6 / 54.6 | **81.1 / 43.2** | 1.40 / 1.60 | 0.70 / 0.80 | 0.43 / 0.43 |
| 961671 | MRA | **64.3 / 41.4** | 73.3 / 52.9 | 1.40 / 1.40 | 0.70 / 0.70 | 0.43 / 0.43 |
| 962005 | MRA | 62.9 / 56.0 | 76.5 / 69.3 | 0.50 / 0.50 | 0.50 / 0.50 | 0.35 / 0.35 |
| 963218 | MRA | 61.9 / 54.3 | 69.6 / 69.7 | 0.52 / 0.52 | 0.52 / 0.52 | 0.37 / 0.43 |
| 963661 | MRA | 64.3 / 59.5 | 80.3 / 52.2 | 0.50 / 0.50 | 0.50 / 0.50 | 0.35 / 0.35 |
| 966396 | MRA | 57.7 / 55.9 | 74.3 / 68.6 | 0.50 / 0.50 | 0.50 / 0.50 | 0.26 / 0.35 |
| 968599 | MRA | 57.0 / 55.1 | 83.8 / 70.1 | 0.55 / 0.55 | 0.55 / 0.55 | 0.50 / 0.50 |
| unknown | MRA | 60.7 / 55.7 | 79.8 / 69.0 | 1.00 / 0.85 | 0.65 / 0.70 | 0.43 / 0.43 |


In [ ]:
loc = pd.read_csv(os.path.join(COMP, "train_localizers.csv"))
LOC_COLS = [c for c in train.columns if c not in ("SeriesInstanceUID", "PatientAge", "PatientSex", "Modality", "Aneurysm Present")]
t = train[train["Modality"].isin(["CTA", "MRA"])].copy()
t["n_loc_train"] = t[LOC_COLS].sum(axis=1)
t["n_loc_points"] = t["SeriesInstanceUID"].map(loc.groupby("SeriesInstanceUID").size()).fillna(0).astype(int)

print("=== consistenta etichete (CTA + MRA)")
print("pozitive fara nicio locatie in train.csv:", ((t["Aneurysm Present"] == 1) & (t["n_loc_train"] == 0)).sum())
print("negative cu locatie in train.csv:        ", ((t["Aneurysm Present"] == 0) & (t["n_loc_train"] > 0)).sum())
print("pozitive fara punct in localizers:       ", ((t["Aneurysm Present"] == 1) & (t["n_loc_points"] == 0)).sum())
print("negative cu punct in localizers:         ", ((t["Aneurysm Present"] == 0) & (t["n_loc_points"] > 0)).sum())
marked = t.set_index("SeriesInstanceUID")[LOC_COLS]
lp = loc[loc["SeriesInstanceUID"].isin(marked.index)]
print("puncte cu locatie nemarcata in train.csv: ", sum(marked.at[u, l] != 1 for u, l in zip(lp["SeriesInstanceUID"], lp["location"])))

print("\n=== anevrisme pe serie")
pos = t[t["Aneurysm Present"] == 1]
print(pd.crosstab(pos["Modality"], pos["n_loc_points"].clip(upper=4).astype(str).replace({"4": "4+"}), margins=True))

print("\n=== serii pozitive pe locatie (o serie poate avea mai multe locatii)")
dist = t.groupby("Modality")[LOC_COLS].sum().T
dist["total"] = dist.sum(axis=1)
dist["pct"] = (dist["total"] / dist["total"].sum() * 100).round(1)
print(dist.sort_values("total", ascending=False).to_string())

## Etichete si localizari (CTA + MRA)

Celula verifica daca cele trei surse de etichete spun acelasi lucru si descrie anevrismele din seriile pozitive:
- `Aneurysm Present` din `train.csv` (eticheta seriei);
- cele 13 coloane de locatie din `train.csv` (0/1 pe locatie);
- punctele din `train_localizers.csv` (cate un punct pentru fiecare anevrism, cu felia, coordonatele si locatia).

### Consistenta etichetelor

| Verificare | Serii |
|---|---|
| Pozitive fara nicio locatie in `train.csv` | 0 |
| Negative cu locatie in `train.csv` | 0 |
| Pozitive fara punct in `train_localizers.csv` | 0 |
| Negative cu punct in `train_localizers.csv` | 0 |
| Puncte cu o locatie nemarcata in `train.csv` | 1 |

### Anevrisme pe serie (doar pozitive)

| Modalitate | 1 | 2 | 3 | 4+ | Total |
|---|---|---|---|---|---|
| CTA | 783 | 145 | 36 | 9 | 973 |
| MRA | 479 | 55 | 15 | 6 | 555 |
| Total | 1262 | 200 | 51 | 15 | 1528 |

### Serii pozitive pe locatie

O serie poate avea anevrisme la mai multe locatii, deci suma pe locatii (1816) depaseste numarul de serii pozitive (1528).

| Locatie | CTA | MRA | Total | % |
|---|---|---|---|---|
| Anterior Communicating Artery | 232 | 81 | 313 | 17.2 |
| Left Supraclinoid ICA | 145 | 123 | 268 | 14.8 |
| Right Middle Cerebral Artery | 161 | 80 | 241 | 13.3 |
| Right Supraclinoid ICA | 104 | 121 | 225 | 12.4 |
| Left Middle Cerebral Artery | 130 | 56 | 186 | 10.2 |
| Right Posterior Communicating Artery | 73 | 20 | 93 | 5.1 |
| Basilar Tip | 65 | 25 | 90 | 5.0 |
| Other Posterior Circulation | 65 | 25 | 90 | 5.0 |
| Right Infraclinoid ICA | 46 | 36 | 82 | 4.5 |
| Left Posterior Communicating Artery | 62 | 13 | 75 | 4.1 |
| Left Infraclinoid ICA | 33 | 32 | 65 | 3.6 |
| Right Anterior Cerebral Artery | 35 | 12 | 47 | 2.6 |
| Left Anterior Cerebral Artery | 32 | 9 | 41 | 2.3 |

